# pivotai — Curriculum-Trained Model (`pivotai-curriculum`)
**Session 3 of 3** — Two sequential training stages on the **same model object**.

- **Stage 1** (lr=2e-4, 2 epochs): Phase 1 domain knowledge — 4,750 records, seq_len ≤ 512
- **Stage 2** (lr=5e-5, 3 epochs): Phase 2 agent reasoning — 450 records, seq_len ≤ 16384

The 4× lower LR in Stage 2 prevents catastrophic forgetting of Stage 1 weights.
Stage 1 checkpoint saved to local storage for crash recovery.

**Platform**: Lightning.ai A100 (40GB VRAM, 200GB disk)

**Actual results**: Stage 1 train_loss=0.313 (final loss 0.241), Stage 2 train_loss=0.686 (final loss 0.505)

**Before running:**
1. Upload `pivotai.zip` to Lightning.ai studio storage
2. Paste HF write token in Cell 7
3. Expected time: ~13 min total (Stage 1 ~8 min, Stage 2 ~5 min)

In [ ]:
# Cell 0 — Set memory env var FIRST (must run before any torch import)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('✓ CUDA alloc config set')

In [ ]:
# Cell 1 — Setup Lightning.ai paths + unzip project
import zipfile, os, sys

BASE = '/teamspace/studios/this_studio'
if not os.path.exists(f'{BASE}/travel_project'):
    with zipfile.ZipFile(f'{BASE}/pivotai.zip', 'r') as z:
        z.extractall(BASE)
    print('✓ Unzipped')
else:
    print('✓ Already extracted')

os.makedirs(f'{BASE}/pivotai_models', exist_ok=True)
print(f'Working directory: {BASE}')

In [ ]:
# Cell 2 — Add project to path + verify all 4 training files
import sys
sys.path.insert(0, f'{BASE}/travel_project')

STAGE1_FILE = f'{BASE}/travel_project/data/training/curriculum_stage1.jsonl'
STAGE2_FILE = f'{BASE}/travel_project/data/training/curriculum_stage2.jsonl'
VAL1_FILE   = f'{BASE}/travel_project/data/training/ft_val.jsonl'
VAL2_FILE   = f'{BASE}/travel_project/data/training/distill_val.jsonl'

for f in [STAGE1_FILE, STAGE2_FILE, VAL1_FILE, VAL2_FILE]:
    lines = open(f).read().splitlines()
    print(f'{f.split("/")[-1]}: {len(lines)} records')

In [ ]:
# Cell 3 — Install dependencies (~5 min)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes -q
print('✓ Install complete')

In [ ]:
# Cell 4 — Load Llama 3.1 8B with 4-bit QLoRA
# Stage 1 filters to 512 tokens, Stage 2 filters to 16384 (full reasoning chains)
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 16384

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-bnb-4bit',
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('✓ Model loaded with MAX_SEQ_LEN=16384 (A100 bf16)')

In [ ]:
# Cell 5 — Prepare both datasets with token-length filtering
from datasets import load_dataset

ALPACA_PROMPT = """### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n{}"""
EOS = tokenizer.eos_token

def format_alpaca(examples):
    return {'text': [
        ALPACA_PROMPT.format(i, inp, out) + EOS
        for i, inp, out in zip(examples['instruction'], examples['input'], examples['output'])
    ]}

def make_filter(max_len):
    def filter_length(examples):
        lengths = tokenizer(examples['text'], truncation=False, return_length=True, padding=False)['length']
        return [l <= max_len for l in lengths]
    return filter_length

# Stage 1: Phase 1 data, filter to 512 tokens (short structured JSON)
stage1_train = load_dataset('json', data_files=STAGE1_FILE, split='train').map(format_alpaca, batched=True)
val1         = load_dataset('json', data_files=VAL1_FILE,   split='train').map(format_alpaca, batched=True)
before = len(stage1_train)
stage1_train = stage1_train.filter(make_filter(512), batched=True)
print(f'✓ Stage 1 train: {before} → {len(stage1_train)} (dropped {before - len(stage1_train)} > 512 tokens)')

# Stage 2: full reasoning chains, filter to 16384 tokens
stage2_train = load_dataset('json', data_files=STAGE2_FILE, split='train').map(format_alpaca, batched=True)
val2         = load_dataset('json', data_files=VAL2_FILE,   split='train').map(format_alpaca, batched=True)
before = len(stage2_train)
stage2_train = stage2_train.filter(make_filter(16384), batched=True)
print(f'✓ Stage 2 train: {before} → {len(stage2_train)} (dropped {before - len(stage2_train)} > 16384 tokens)')

In [ ]:
# Cell 6 — Curriculum training: Stage 1 then Stage 2 on the SAME model object
from trl import SFTTrainer
from transformers import TrainingArguments

COMMON_ARGS = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    fp16=False,
    bf16=True,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    save_strategy='no',
    report_to='none',
    dataloader_num_workers=0,
)

# ── Stage 1: Domain knowledge (Phase 1, lr=2e-4, 2 epochs) ──────────────────
print('=== STAGE 1: Domain knowledge (Phase 1 data, lr=2e-4) ===')

trainer_s1 = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=stage1_train,
    dataset_text_field='text',
    max_seq_length=512,
    args=TrainingArguments(
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_steps=20,
        logging_steps=50,
        output_dir=f'{BASE}/curriculum_s1',
        **COMMON_ARGS,
    ),
)

stats_s1 = trainer_s1.train()
print(f'Stage 1 complete — {stats_s1.metrics["train_runtime"]:.0f}s, loss={stats_s1.metrics["train_loss"]:.4f}')

# Save Stage 1 checkpoint for crash recovery
model.save_pretrained(f'{BASE}/pivotai_models/curriculum_stage1_lora')
tokenizer.save_pretrained(f'{BASE}/pivotai_models/curriculum_stage1_lora')
print('✓ Stage 1 checkpoint saved')

# ── Stage 2: Reasoning on top (Phase 2, lr=5e-5, 3 epochs) ──────────────────
# SAME model object — Stage 1 LoRA weights remain in memory
# 4x lower LR is critical to prevent catastrophic forgetting of Stage 1
print('\n=== STAGE 2: Agent reasoning (Phase 2 traces, lr=5e-5) ===')

trainer_s2 = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=stage2_train,
    dataset_text_field='text',
    max_seq_length=16384,
    args=TrainingArguments(
        num_train_epochs=3,
        learning_rate=5e-5,
        warmup_steps=5,
        logging_steps=20,
        output_dir=f'{BASE}/curriculum_s2',
        **COMMON_ARGS,
    ),
)

stats_s2 = trainer_s2.train()
print(f'Stage 2 complete — {stats_s2.metrics["train_runtime"]:.0f}s, loss={stats_s2.metrics["train_loss"]:.4f}')

total = stats_s1.metrics['train_runtime'] + stats_s2.metrics['train_runtime']
print(f'\n✓ Total curriculum training: {total:.0f}s ({total/3600:.1f}h)')

In [ ]:
# Cell 7 — Save LoRA, convert to GGUF, upload to HuggingFace
import shutil
from huggingface_hub import HfApi

HF_TOKEN    = ''  # paste your HF write token here
HF_USERNAME = 'ishreyadev'

# Step 1: Convert to GGUF locally (NOT remote storage — buffering causes 0-byte errors)
model.save_pretrained_gguf(f'{BASE}/pivotai_curriculum', tokenizer, quantization_method='q4_k_m')
gguf_src = f'{BASE}/pivotai_curriculum_gguf/Meta-Llama-3.1-8B.Q4_K_M.gguf'
shutil.copy(gguf_src, f'{BASE}/pivotai_models/pivotai_curriculum.gguf')
print('✓ GGUF created and copied')

# Step 2: Upload LoRA to HF
model.push_to_hub(f'{HF_USERNAME}/pivotai-curriculum-lora', token=HF_TOKEN, private=False)
tokenizer.push_to_hub(f'{HF_USERNAME}/pivotai-curriculum-lora', token=HF_TOKEN)
print(f'✓ LoRA → huggingface.co/{HF_USERNAME}/pivotai-curriculum-lora')

# Step 3: Upload GGUF to HF
api = HfApi()
api.create_repo(f'{HF_USERNAME}/pivotai-curriculum-gguf', token=HF_TOKEN, exist_ok=True)
api.upload_file(
    path_or_fileobj=gguf_src,
    path_in_repo='pivotai_curriculum.Q4_K_M.gguf',
    repo_id=f'{HF_USERNAME}/pivotai-curriculum-gguf',
    token=HF_TOKEN,
)
print(f'✓ GGUF → huggingface.co/{HF_USERNAME}/pivotai-curriculum-gguf')
print('\npivotai-curriculum COMPLETE')